# 08 · Classes, OOP and dataclasses

After this notebook you can write a class with the right mix of instance and class attributes, properties, class/static methods and dunder methods; choose between inheritance, composition, abstract base classes and Protocols; use dataclasses properly — and build the `Message` / `Conversation` / `Tool` classes that every agent framework is shaped around.

**Time:** ~40 min · **Runs:** offline on CPU in < 1 min · **Needs:** [06 · Functions](06_functions_args_scope_lambda.ipynb), [07 · Iterators and generators](07_comprehensions_iterators_generators.ipynb)

## Why this matters in interviews

- OOP questions are standard warm-ups: `self`, class vs instance attributes, `@classmethod` vs `@staticmethod`, `__repr__` vs `__str__`, `super()` and the MRO.
- Every agent framework is a set of small classes — messages, tools, retrievers, memories — and "design the classes for X" is a common take-home and live-coding prompt.
- Knowing *when not* to inherit (composition, Protocols) and when a `@dataclass` replaces thirty lines of boilerplate is a senior signal.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag04` | What makes a good tool definition? |

## What interviewers ask

- What is `self`? Is `__init__` the constructor?
- Class attributes vs instance attributes — what goes wrong with a mutable class attribute?
- `@classmethod` vs `@staticmethod` vs a normal method?
- `__repr__` vs `__str__`? Why does defining `__eq__` make your objects unhashable?
- How do `super()` and the MRO work with multiple inheritance?
- Composition vs inheritance — how do you choose?
- Abstract base classes vs `typing.Protocol` (duck typing)?
- What does `@dataclass` generate? What are `field(default_factory=...)`, `frozen=True`, `order=True` and `slots=True` for?

## 1 · A class, `__init__` and `self`

A class is a blueprint; calling it creates an **instance**. Python first creates the empty object (`__new__`) and then calls `__init__(self, ...)` to fill in its attributes — so `__init__` is really the *initialiser*, not the constructor. `self` is simply the instance, passed automatically as the first argument: `doc.word_count()` is the same call as `Document.word_count(doc)`.

In [ ]:
class Document:
    def __init__(self, text, source):
        self.text = text                   # instance attributes: one set per object
        self.source = source

    def word_count(self):
        return len(self.text.split())

    def preview(self, n=20):
        return self.text[:n] + ("…" if len(self.text) > n else "")

doc = Document("Refunds are processed within five business days.", source="faq.md")
print(doc.word_count(), "|", Document.word_count(doc), "|", doc.preview())
print("instance __dict__:", doc.__dict__)
assert doc.word_count() == Document.word_count(doc) == 7

## 2 · Instance vs class attributes — and the shared-list bug

Attributes assigned on `self` belong to one instance. Attributes assigned in the class body belong to the **class** and are shared by every instance — right for constants and defaults, wrong for per-instance mutable state. Reading `obj.attr` looks on the instance first, then on the class; *assigning* `obj.attr = ...` always creates an instance attribute, shadowing the class one.

**Predict, then run:** two separate chats get one message each. How many messages does `chat_b` have?

In [ ]:
class ChatBuggy:
    max_messages = 50                # class attribute: a shared constant — fine
    messages = []                    # class attribute: ONE list shared by every chat — bug

    def add(self, text):
        self.messages.append(text)   # finds the class list and mutates it

chat_a, chat_b = ChatBuggy(), ChatBuggy()
chat_a.add("hello from A")
chat_b.add("hello from B")
print("chat_b.messages:", chat_b.messages, "| shared?", chat_a.messages is chat_b.messages)

class Chat:
    max_messages = 50
    def __init__(self):
        self.messages = []           # instance attribute: a new list per chat

    def add(self, text):
        self.messages.append(text)

c1, c2 = Chat(), Chat()
c1.add("hello from 1")
c2.max_messages = 10                 # creates an instance attribute on c2 only
print("c2.messages:", c2.messages, "| max_messages c1, c2, class:", c1.max_messages, c2.max_messages, Chat.max_messages)
assert chat_b.messages == ["hello from A", "hello from B"] and c2.messages == [] and Chat.max_messages == 50

## 3 · `@property`: computed and validated attributes

A property looks like a plain attribute to the caller but runs a method. Use it for **computed** values and for **validation** on assignment (a temperature must be between 0 and 2) — without changing the public interface from simple attribute access. A property without a setter is read-only.

In [ ]:
class GenerationConfig:
    def __init__(self, model, temperature=0.7):
        self.model = model
        self.temperature = temperature            # goes through the setter below

    @property
    def temperature(self):
        return self._temperature

    @temperature.setter
    def temperature(self, value):
        if not 0.0 <= value <= 2.0:
            raise ValueError(f"temperature must be in [0, 2], got {value}")
        self._temperature = float(value)

    @property
    def is_deterministic(self):                   # read-only, computed
        return self._temperature == 0.0

cfg = GenerationConfig("gpt-4.1-mini")
cfg.temperature = 0
print("temperature:", cfg.temperature, "| deterministic:", cfg.is_deterministic)
try:
    cfg.temperature = 5
except ValueError as err:
    print("ValueError:", err)
try:
    cfg.is_deterministic = False
except AttributeError as err:
    print("AttributeError:", err)
assert cfg.is_deterministic and cfg.temperature == 0.0

## 4 · `@classmethod` and `@staticmethod`

| Kind | First argument | Use it for |
|---|---|---|
| instance method | `self`, the instance | behaviour that uses the object's data |
| `@classmethod` | `cls`, the class | **alternate constructors** — `from_dict`, `from_json` — that also work for subclasses |
| `@staticmethod` | nothing | a helper that belongs with the class but needs neither |

(The `__repr__` below makes the objects print readably; section 5 covers it.)

In [ ]:
import json

class Chunk:
    def __init__(self, text, doc_id, position=0):
        self.text, self.doc_id, self.position = self.normalise(text), doc_id, position

    @classmethod
    def from_dict(cls, data):                      # cls, not Chunk: a subclass gets its own type back
        return cls(data["text"], data["doc_id"], data.get("position", 0))

    @classmethod
    def from_json(cls, line):
        return cls.from_dict(json.loads(line))

    @staticmethod
    def normalise(text):
        return " ".join(text.split())

    def __repr__(self):
        return f"{type(self).__name__}({self.text!r}, doc_id={self.doc_id!r}, position={self.position})"

class CodeChunk(Chunk):
    pass

print(Chunk.from_json('{"text": "  Refunds   take 5 days. ", "doc_id": "faq"}'))
print(CodeChunk.from_dict({"text": "def f(): pass", "doc_id": "utils.py", "position": 3}))
print(Chunk.normalise("  a   b "))
assert type(CodeChunk.from_dict({"text": "x", "doc_id": "y"})) is CodeChunk

## 5 · Dunder methods: making objects behave like built-ins

"Dunder" (double-underscore) methods are hooks Python calls for you: `print(x)` → `__str__`; the REPL and containers → `__repr__`; `==` → `__eq__`; `hash(x)`, dict keys and sets → `__hash__`; `len(x)` → `__len__`; `<`, and therefore `sorted` / `max` → `__lt__`; `x(...)` → `__call__`.

Rules worth knowing:

- `__repr__` is for developers (unambiguous, ideally looks like the constructor call); `__str__` is for end users. `print` uses `__str__`, but **containers show the `__repr__`** of their items.
- Defining `__eq__` sets `__hash__` to `None` — instances become unhashable — unless you also define `__hash__`, because equal objects must hash equal. Hash exactly the fields that define equality, and only if they never change.
- Return `NotImplemented` from `__eq__` for types you don't handle; `functools.total_ordering` fills in `<=`, `>`, `>=` from `__eq__` and `__lt__`.

**Predict, then run:** what do `print(best)` and `print([best])` show?

In [ ]:
from functools import total_ordering

@total_ordering
class ScoredChunk:
    def __init__(self, text, score):
        self.text, self.score = text, score

    def __repr__(self):
        return f"ScoredChunk({self.text!r}, score={self.score})"

    def __str__(self):
        return f"[{self.score:.2f}] {self.text}"

    def __eq__(self, other):
        if not isinstance(other, ScoredChunk):
            return NotImplemented
        return (self.text, self.score) == (other.text, other.score)

    def __hash__(self):
        return hash((self.text, self.score))

    def __lt__(self, other):
        return self.score < other.score

    def __len__(self):
        return len(self.text)

hits = [ScoredChunk("refunds take 5 days", 0.91), ScoredChunk("shipping is free", 0.35),
        ScoredChunk("refunds take 5 days", 0.91), ScoredChunk("contact support", 0.62)]
best = max(hits)                                      # uses __lt__
print(best)
print([best])
print("sorted, best first:", [h.score for h in sorted(hits, reverse=True)])
print("unique hits:", len(set(hits)), "| len(best):", len(best), "| best >= hits[1]:", best >= hits[1])

class OnlyEq:
    def __init__(self, x):
        self.x = x
    def __eq__(self, other):
        return isinstance(other, OnlyEq) and self.x == other.x
print("OnlyEq.__hash__ is None:", OnlyEq.__hash__ is None)
try:
    {OnlyEq(1)}
except TypeError as err:
    print("TypeError:", err)
assert len(set(hits)) == 3 and best == hits[0] and best > hits[1]

### `__call__`: objects you can call like functions

An object with `__call__` can hold configuration *and* be used wherever a function is expected — a callable prompt template, a tokenizer, a model wrapper. PyTorch's `nn.Module` works this way: `model(x)` calls `model.__call__`, which runs `forward`.

In [ ]:
class PromptTemplate:
    def __init__(self, template):
        self.template = template
        self.calls = 0

    def __call__(self, **fields):
        self.calls += 1
        return self.template.format(**fields)

    def __repr__(self):
        return f"PromptTemplate({self.template!r})"

summarise = PromptTemplate("Summarise in {n} words:\n{text}")
print(summarise(n=10, text="Refunds are processed within five business days."))
print("callable?", callable(summarise), "| calls so far:", summarise.calls, "|", summarise)
assert summarise(n=5, text="x").startswith("Summarise in 5 words") and summarise.calls == 2

## 6 · Inheritance, `super()` and the MRO

A subclass inherits everything from its base class and can **override** methods; `super().method(...)` calls the next implementation up the chain — typically to *extend* `__init__` or a method rather than replace it.

With multiple inheritance, Python linearises the class graph into the **method resolution order** (MRO, computed by the C3 algorithm): the order in which classes are searched for an attribute. `super()` means "the next class in the MRO **of the object's class**", not "my parent" — which is why, in a diamond, every class's method runs exactly once.

```
          Base
         /    \
    Logging  Caching
         \    /
       SmartClient        MRO: SmartClient → Logging → Caching → Base → object
```

In [ ]:
class Base:
    def handle(self, request):
        return ["Base"]

class Logging(Base):
    def handle(self, request):
        return ["Logging"] + super().handle(request)

class Caching(Base):
    def handle(self, request):
        return ["Caching"] + super().handle(request)

class SmartClient(Logging, Caching):
    def handle(self, request):
        return ["SmartClient"] + super().handle(request)

print("call chain:", " -> ".join(SmartClient().handle("q")))
print("MRO       :", [cls.__name__ for cls in SmartClient.__mro__])
print("Logging() on its own:", Logging().handle("q"), "<- same method; here super() goes straight to Base")
print("isinstance:", isinstance(SmartClient(), Caching), "| issubclass:", issubclass(SmartClient, Base))
assert SmartClient().handle("q") == ["SmartClient", "Logging", "Caching", "Base"]

## 7 · Composition over inheritance

Inheritance says "**is a**"; composition says "**has a**". Deep hierarchies (`OpenAIRAGWithBM25AndReranker`) break every time one part changes. Composing small parts — a pipeline that *has* a retriever and *has* a generator, each swappable — is how RAG and agent frameworks are actually structured, and it makes testing easy: pass in a fake.

In [ ]:
DOCS = ["refunds take five business days", "shipping is free over 50 dollars", "support is open on weekdays"]

class KeywordRetriever:
    def __init__(self, docs, top_k=2):
        self.docs, self.top_k = docs, top_k

    def retrieve(self, query):
        terms = set(query.lower().split())
        return sorted(self.docs, key=lambda d: -len(terms & set(d.split())))[: self.top_k]

class EchoGenerator:
    """Stands in for an LLM: 'answers' with the first retrieved passage."""
    def generate(self, question, passages):
        return f"Based on '{passages[0]}': ..." if passages else "I don't know."

class RAGPipeline:
    def __init__(self, retriever, generator):          # has-a retriever, has-a generator
        self.retriever, self.generator = retriever, generator

    def answer(self, question):
        return self.generator.generate(question, self.retriever.retrieve(question))

rag = RAGPipeline(KeywordRetriever(DOCS), EchoGenerator())
print(rag.answer("how many days do refunds take"))

class FixedRetriever:                                  # swap one part without touching the pipeline
    def retrieve(self, query):
        return ["(a test passage)"]
print(RAGPipeline(FixedRetriever(), EchoGenerator()).answer("anything"))
assert "refunds take five business days" in rag.answer("refunds days")

## 8 · Abstract base classes: the `Tool` interface (`ag04`)

An **abstract base class** declares methods that subclasses *must* implement (`@abstractmethod`). You cannot instantiate the ABC itself, or a subclass that forgot a method — the error comes at construction time, not later in production. It is the natural shape for a plugin interface such as an agent's tools.

The base class below also implements what every tool shares, following the rules of a good tool definition: a precise **name**; a **description** that says when to use it (and when not); a JSON Schema for its **parameters**, with enums so wrong values are impossible rather than discouraged; and **validation in code**, so a bad argument from the model is rejected with a clear message — which you send back to the model so it can correct itself — instead of being executed. `to_openai_tool()` emits the `tools=[...]` entry the Chat Completions API expects.

In [ ]:
from abc import ABC, abstractmethod

class Tool(ABC):
    name: str
    description: str
    parameters: dict                                 # JSON Schema for the arguments

    @abstractmethod
    def run(self, **kwargs) -> str:
        """Do the work. Arguments have already been validated."""

    def validate(self, args: dict) -> None:
        props, required = self.parameters["properties"], self.parameters.get("required", [])
        missing = [k for k in required if k not in args]
        unknown = [k for k in args if k not in props]
        if missing or unknown:
            raise ValueError(f"{self.name}: missing {missing}, unknown {unknown}")
        py_types = {"string": str, "integer": int, "number": (int, float)}
        for key, value in args.items():
            spec = props[key]
            if isinstance(value, bool) or not isinstance(value, py_types[spec["type"]]):    # bool is an int!
                raise ValueError(f"{self.name}: {key!r} must be {spec['type']}, got {value!r}")
            if "enum" in spec and value not in spec["enum"]:
                raise ValueError(f"{self.name}: {key!r} must be one of {spec['enum']}, got {value!r}")

    def __call__(self, arguments_json: str) -> str:
        args = json.loads(arguments_json)            # models send arguments as a JSON *string*
        self.validate(args)
        return self.run(**args)

    def to_openai_tool(self) -> dict:
        return {"type": "function", "function": {
            "name": self.name, "description": self.description,
            "parameters": {**self.parameters, "additionalProperties": False}}}

try:
    Tool()
except TypeError as err:
    print("TypeError:", err)

In [ ]:
import operator

class CalculatorTool(Tool):
    name = "calculator"
    description = "Exact arithmetic on two numbers. Use it for every calculation instead of doing maths yourself."
    parameters = {"type": "object",
                  "properties": {"a": {"type": "number"}, "b": {"type": "number"},
                                 "op": {"type": "string", "enum": ["add", "sub", "mul", "div"]}},
                  "required": ["a", "b", "op"]}
    OPS = {"add": operator.add, "sub": operator.sub, "mul": operator.mul, "div": operator.truediv}

    def run(self, a, b, op):
        if op == "div" and b == 0:
            return "error: division by zero"
        return str(self.OPS[op](a, b))

class SearchTool(Tool):
    name = "search_docs"
    description = "Keyword search over help-centre articles. Use it for policy questions, not for maths."
    parameters = {"type": "object",
                  "properties": {"query": {"type": "string"}, "top_k": {"type": "integer"}},
                  "required": ["query"]}

    def __init__(self, docs):
        self.retriever = KeywordRetriever(docs)          # composition again

    def run(self, query, top_k=1):
        self.retriever.top_k = top_k
        return " | ".join(self.retriever.retrieve(query))

class BrokenTool(Tool):                                  # forgot to implement run()
    name, description, parameters = "broken", "...", {"type": "object", "properties": {}}

try:
    BrokenTool()
except TypeError as err:
    print("TypeError:", err)

In [ ]:
registry = {tool.name: tool for tool in (CalculatorTool(), SearchTool(DOCS))}

model_tool_calls = [                                      # the shape of message.tool_calls in a chat response
    {"id": "call_1", "function": {"name": "calculator", "arguments": '{"a": 1299, "b": 3, "op": "mul"}'}},
    {"id": "call_2", "function": {"name": "search_docs", "arguments": '{"query": "refunds days"}'}},
    {"id": "call_3", "function": {"name": "calculator", "arguments": '{"a": 2, "b": 3, "op": "pow"}'}},
    {"id": "call_4", "function": {"name": "search_docs", "arguments": '{"query": "refunds", "top_k": true}'}},
    {"id": "call_5", "function": {"name": "send_email", "arguments": '{}'}},
]
tool_messages = []
for call in model_tool_calls:
    fn = call["function"]
    try:
        output = registry[fn["name"]](fn["arguments"])
    except KeyError:
        output = f"error: unknown tool {fn['name']!r}"
    except ValueError as err:                             # includes json.JSONDecodeError
        output = f"error: {err}"                          # report it back to the model; don't crash
    tool_messages.append({"role": "tool", "tool_call_id": call["id"], "content": output})
    print(f"{call['id']}: {output}")

print(json.dumps(registry["calculator"].to_openai_tool(), indent=2))
assert tool_messages[0]["content"] == "3897" and "must be one of" in tool_messages[2]["content"]
assert "must be integer" in tool_messages[3]["content"] and "unknown tool" in tool_messages[4]["content"]

## 9 · `@dataclass`: classes for data, without the boilerplate

`@dataclass` reads the annotated class attributes and generates `__init__`, `__repr__` and `__eq__` for you — and more on request:

| Option | Generates / does |
|---|---|
| (default) | `__init__`, `__repr__`, `__eq__` from the fields, in order |
| `field(default_factory=list)` | a fresh default per instance; a plain `= []` is **refused**, which prevents the shared-list bug of section 2 |
| `frozen=True` | assignment raises `FrozenInstanceError`; together with `eq` you also get `__hash__`, so instances can be dict keys |
| `order=True` | `<`, `<=`, `>`, `>=` comparing the fields as a tuple, in definition order |
| `slots=True` (3.10+) | `__slots__` (section 10) |
| `kw_only=True` (3.10+) | every field must be passed by keyword |
| `__post_init__` | your hook for validation after the generated `__init__` |

Dataclasses do **not** validate types at runtime — annotations are just declarations. For untrusted data (model output, API payloads) use Pydantic ([13](13_type_hints_and_pydantic.ipynb)).

In [ ]:
from dataclasses import FrozenInstanceError, asdict, dataclass, field, replace

try:
    @dataclass
    class BadConversation:
        messages: list = []
except ValueError as err:
    print("ValueError:", err)

@dataclass(frozen=True)
class CacheKey:
    model: str
    prompt: str
    temperature: float = 0.0

@dataclass(order=True)
class Ranked:
    score: float
    doc_id: str = field(compare=False)                # left out of ordering and equality

cache = {CacheKey("gpt-4.1-mini", "What is RAG?"): "cached answer"}
print("frozen key lookup:", cache[CacheKey("gpt-4.1-mini", "What is RAG?")])
key = CacheKey("m", "p")
print("repr:", key)
try:
    key.temperature = 1.0
except FrozenInstanceError as err:
    print("FrozenInstanceError:", err)
print("replace() makes a changed copy:", replace(key, temperature=1.0))
print("order=True:", sorted([Ranked(0.4, "a"), Ranked(0.9, "b"), Ranked(0.7, "c")], reverse=True))
assert CacheKey("m", "p") == CacheKey("m", "p", 0.0) and Ranked(0.5, "x") == Ranked(0.5, "y")

### Practical: `Message` and `Conversation`

These two dataclasses are the core of every chat application and agent framework (LangChain's message classes and LangGraph's state are richer versions of the same idea). `Message` is frozen and validates its role in `__post_init__`; `Conversation` keeps the system prompt separate from the turns, owns a list created per instance with `default_factory`, and converts itself to the list-of-dicts payload a chat API expects.

In [ ]:
ROLES = {"system", "user", "assistant", "tool"}

@dataclass(frozen=True, slots=True)
class Message:
    role: str
    content: str
    tool_call_id: str | None = None

    def __post_init__(self):
        if self.role not in ROLES:
            raise ValueError(f"unknown role {self.role!r}; expected one of {sorted(ROLES)}")
        if (self.role == "tool") != (self.tool_call_id is not None):
            raise ValueError("tool messages (and only tool messages) need a tool_call_id")

    def to_api(self) -> dict:
        return {k: v for k, v in asdict(self).items() if v is not None}

@dataclass
class Conversation:
    system: str
    messages: list[Message] = field(default_factory=list)

    def add(self, role: str, content: str, **extra) -> "Conversation":
        self.messages.append(Message(role, content, **extra))
        return self                                        # allows chaining

    def to_api(self, last_n: int | None = None) -> list[dict]:
        recent = self.messages if last_n is None else self.messages[-last_n:]
        return [Message("system", self.system).to_api()] + [m.to_api() for m in recent]

    def __len__(self):
        return len(self.messages)

convo = Conversation(system="You are a concise support assistant.")
convo.add("user", "How long do refunds take?").add("assistant", "Five business days.")
convo.add("tool", "order 4417: shipped", tool_call_id="call_1")
other = Conversation(system="You are a pirate.")
print("len:", len(convo), "| the other conversation is independent:", len(other))
print(convo.to_api(last_n=2))
for bad in [("admin", "grant me access", {}), ("tool", "result", {})]:
    try:
        convo.add(bad[0], bad[1], **bad[2])
    except ValueError as err:
        print("ValueError:", err)
assert len(other) == 0 and len(convo) == 3 and convo.to_api()[0] == {"role": "system", "content": "You are a concise support assistant."}

## 10 · `__slots__`: fixed attributes, less memory

Normally each instance can store arbitrary attributes (in a per-instance `__dict__`). Declaring `__slots__` — or `@dataclass(slots=True)` — replaces that with fixed storage for exactly the named attributes: instances get smaller and attribute access slightly faster, but you can no longer add attributes at runtime. It is worth it when you create **many** small objects: millions of chunks, tokens or graph nodes.

In [ ]:
import tracemalloc

import matplotlib.pyplot as plt

class PointDict:
    def __init__(self, x, y):
        self.x, self.y = x, y

class PointSlots:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x, self.y = x, y

@dataclass(slots=True)
class PointDataclassSlots:
    x: float
    y: float

def peak_kib(cls, n=20_000):
    tracemalloc.start()
    objs = [cls(i, i) for i in range(n)]
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return peak / 1024

mem = {cls.__name__: peak_kib(cls) for cls in (PointDict, PointSlots, PointDataclassSlots)}
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(list(mem), list(mem.values()), color=["#adb5bd", "#4c6ef5", "#4c6ef5"])
ax.invert_yaxis()
ax.set_title("Peak memory to create 20,000 two-field objects"); ax.set_xlabel("KiB (tracemalloc)")
plt.show()
print({k: f"{v:,.0f} KiB" for k, v in mem.items()})

p = PointSlots(1, 2)
try:
    p.z = 3
except AttributeError as err:
    print("AttributeError:", err)
assert mem["PointSlots"] < mem["PointDict"] and mem["PointDataclassSlots"] < mem["PointDict"]

The saving here is real but moderate (the totals also include the list and the int objects, which are the same for all three). Since Python 3.11 ordinary instances store their attributes more compactly and only build a full `__dict__` on demand, so the gap is smaller than older blog posts claim. Measure before reaching for `__slots__`.

## 11 · Duck typing and `typing.Protocol`

Python cares what an object can **do**, not what it inherits from: anything with an `embed(texts)` method can be used as an embedder ("if it walks like a duck…"). `typing.Protocol` writes that expectation down so type checkers (mypy, pyright) can verify it — **structural** typing, no inheritance required. `@runtime_checkable` also enables `isinstance` checks, which only test that the methods *exist*, not their signatures.

| | ABC | Protocol |
|---|---|---|
| Relationship | nominal: you must inherit (or register) | structural: you just need the methods |
| Enforced | at instantiation (missing abstract methods) | by the type checker; `isinstance` only with `@runtime_checkable` |
| Shares code | yes — concrete methods such as `validate` | not intended to |
| Best for | plugin interfaces you own (`Tool`) | accepting objects from code you don't control |

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Embedder(Protocol):
    def embed(self, texts: list[str]) -> list[list[float]]: ...

class CharCountEmbedder:                       # does not inherit from Embedder
    def embed(self, texts):
        return [[len(t), t.count(" ")] for t in texts]

class VowelEmbedder:
    def embed(self, texts):
        return [[sum(ch in "aeiou" for ch in t)] for t in texts]

def index_documents(docs: list[str], embedder: Embedder) -> list[list[float]]:
    return embedder.embed(docs)

for emb in (CharCountEmbedder(), VowelEmbedder()):
    print(f"{type(emb).__name__:<18} -> {index_documents(['refund policy', 'shipping'], emb)} "
          f"| isinstance(emb, Embedder): {isinstance(emb, Embedder)}")
print("is a str an Embedder?", isinstance("text", Embedder))
assert isinstance(CharCountEmbedder(), Embedder) and not isinstance("text", Embedder)

## Try it yourself

**1.** Add a `WordCountTool` (one required string parameter, `text`), register it, and call it through the registry with a JSON argument string. Then show the validation error for a call with no `text`.

In [ ]:
# Solution — try it yourself first, then run this
class WordCountTool(Tool):
    name = "word_count"
    description = "Count the words in a text. Use it when the user asks how long a text is."
    parameters = {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}

    def run(self, text):
        return str(len(text.split()))

registry[WordCountTool.name] = WordCountTool()
print(registry["word_count"]('{"text": "retrieval augmented generation"}'))
try:
    registry["word_count"]("{}")
except ValueError as err:
    print("ValueError:", err)
assert registry["word_count"]('{"text": "a b c"}') == "3"

**2.** Make a frozen dataclass `EvalCase(question, expected)`. Use instances as keys of a results dict, and de-duplicate a list of cases with a set.

In [ ]:
# Solution — try it yourself first, then run this
@dataclass(frozen=True)
class EvalCase:
    question: str
    expected: str

cases = [EvalCase("What is RAG?", "retrieval + generation"), EvalCase("2+2?", "4"), EvalCase("2+2?", "4")]
results = {case: "pass" for case in cases}
print(len(set(cases)), "unique cases |", results[EvalCase("2+2?", "4")])
assert len(set(cases)) == 2 and len(results) == 2

**3.** This class leaks state between sessions. Find the bug and fix it.
```python
class Session:
    tool_calls = []
    def record(self, name):
        self.tool_calls.append(name)
```

In [ ]:
# Solution — try it yourself first, then run this
class Session:
    def __init__(self):
        self.tool_calls = []                 # per-instance list, created in __init__

    def record(self, name):
        self.tool_calls.append(name)

s1, s2 = Session(), Session()
s1.record("search")
print("s1:", s1.tool_calls, "| s2:", s2.tool_calls)
assert s2.tool_calls == []

**4.** Write a `Usage` dataclass (`prompt_tokens`, `completion_tokens`, both defaulting to 0) with a `total` property and an `__add__` method, so a list of usages can be summed with `sum(usages, Usage())`.

In [ ]:
# Solution — try it yourself first, then run this
@dataclass
class Usage:
    prompt_tokens: int = 0
    completion_tokens: int = 0

    @property
    def total(self) -> int:
        return self.prompt_tokens + self.completion_tokens

    def __add__(self, other: "Usage") -> "Usage":
        if not isinstance(other, Usage):
            return NotImplemented
        return Usage(self.prompt_tokens + other.prompt_tokens, self.completion_tokens + other.completion_tokens)

usages = [Usage(120, 45), Usage(300, 80), Usage(50)]
grand = sum(usages, Usage())
print(grand, "| total:", grand.total)
assert grand == Usage(470, 125) and grand.total == 595

## Say it in an interview

- **Basics (30 s):** "`self` is the instance, passed explicitly; `__new__` creates the object and `__init__` initialises it. Instance attributes live on the object; class attributes are shared by all instances — fine for constants, a bug for mutable state like a list of messages, which belongs in `__init__`."
- **Method kinds:** instance methods use `self`; `@classmethod` gets `cls` and is the idiom for alternate constructors that respect subclasses; `@staticmethod` is a namespaced helper. `@property` gives computed or validated attributes behind plain attribute syntax.
- **Dunders:** `__repr__` for developers (containers use it), `__str__` for users; defining `__eq__` removes `__hash__` unless you define it — hash the same immutable fields you compare; `__lt__` makes objects sortable; `__call__` makes them callable.
- **Inheritance:** `super()` follows the MRO (C3 linearisation) of the instance's class, so cooperative methods in a diamond each run once. Prefer composition — a pipeline *has* a retriever — and keep hierarchies shallow.
- **Interfaces:** an ABC enforces required methods at instantiation and can share code (our `Tool.validate`); a `Protocol` is structural — anything with the right methods fits — and is checked by type checkers.
- **Dataclasses:** generated `__init__`/`__repr__`/`__eq__`; `default_factory` for mutable defaults; `frozen=True` for hashable value objects (cache keys); `order=True`; `slots=True` for memory. No runtime type validation — that's Pydantic.
- **Tools (`ag04`):** precise name, a description that says when (not) to use it, a strict schema with enums, validation in code, and errors returned to the model rather than raised to the user.

## Next

- [09 · Errors and exceptions](09_errors_and_exceptions.ipynb) and [13 · Type hints and Pydantic](13_type_hints_and_pydantic.ipynb) — validation that dataclasses don't do.
- [Tool design and function schemas](../15_agentic_ai/02_tool_design_and_function_schemas.ipynb) and [agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb) — the `Tool` class at work.
- [LangChain: models, messages and prompts](../13_langchain/01_models_messages_and_prompts.ipynb) — a framework's version of `Message`.
- [16 · Python gotchas interviewers love](16_python_gotchas_interviewers_love.ipynb).
- Theory: the [python_basics course](../../python_basics/index.html) (chapter 9: files and projects, with an annotated class) · the [interview bank](../../ai_interview_prep/index.html) for `ag04`.